# 커밋과 .gitignore

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.

## 명령을 실행하기 전에 알아둘 Git 상태

Git은 파일을 세 자리에서 봅니다. **작업 폴더(working tree)** 는 지금 편집하는 파일이 있는 자리입니다. **색인(index)** 은 다음 커밋에 넣을 파일 상태를 골라 두는 자리이며, 스테이징 영역(staging area)이라고도 합니다. **커밋(commit)** 은 색인에 고른 상태를 저장소 이력에 기록한 결과입니다. 따라서 `git add app.py`는 파일을 영구 저장하는 명령이 아니라 현재 `app.py`를 색인에 올리는 명령입니다. 이어서 `git commit -m "메시지"`를 실행해야 그 상태가 커밋으로 기록됩니다.

**HEAD** 는 현재 체크아웃한 커밋을 가리킵니다. 보통 HEAD는 `main` 같은 **브랜치(branch)** 이름을 가리키고, 브랜치 이름은 다시 최신 커밋을 가리킵니다. 새 커밋을 만들면 현재 브랜치가 새 커밋으로 이동합니다. 이 절에서는 HEAD가 현재 커밋을 가리킨다는 관계까지만 사용합니다.

명령은 왼쪽부터 `git` → 하위 명령 → 옵션 → 대상으로 읽습니다. 예를 들어 `git status --short`에서 `status`는 상태를 보고, `--short`는 짧은 형식으로 표시하라는 옵션입니다. 코드의 `git("status", "--short", cwd=repo)`는 터미널에서 같은 명령을 실행하도록 인자를 나눈 Python 표현입니다.


## 준비

임시 저장소를 만들고 이 노트북에만 적용되는 작성자 정보를 설정합니다. `git init -b main`에서 `init`은 현재 폴더를 Git 저장소로 만들고, `-b main`은 첫 브랜치 이름을 `main`으로 정합니다. 아직 커밋이 없으므로 브랜치 이름만 준비된 상태입니다.

이 절에서는 `app.py`가 작업 폴더, 색인, 커밋을 차례로 이동하는 모습을 확인합니다. `.env`는 작업 폴더에는 있지만 `.gitignore` 규칙 때문에 색인에 올라가지 않습니다. 두 파일이 같은 폴더에 있어도 Git의 추적 상태는 다를 수 있습니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 저장소를 main 브랜치로 초기화합니다.
repo = root / "orders"
repo.mkdir()
git("init", "-b", "main", cwd=repo)
configure(repo)
print(repo.name, git("branch", "--show-current", cwd=repo))

orders main


## 기본 실행

커밋 메시지 앞의 `feat`, `fix`, `chore`는 이 교재가 변경 목적을 구분하려고 사용한 팀 관례입니다. Git이 요구하는 필수 문법이 아니며 저장소마다 다른 규칙을 사용할 수 있습니다.

무시 규칙을 첫 커밋에, 프로그램을 다음 커밋에 기록합니다. `git add .gitignore`는 그 파일만 색인에 올리고, `git commit -m ...`의 `-m`은 커밋 메시지를 명령줄에서 지정합니다. 다음 `git add app.py`는 새 색인을 구성하므로 두 커밋은 서로 다른 목적을 갖습니다.

실행 뒤 `git status --short --ignored`에서 앞의 `!!`는 무시된 파일, `git log -2 --pretty=%s`의 `-2`는 최근 두 커밋, `%s`는 제목만 출력하라는 뜻입니다. `.env`가 `!!`로 보이고 두 메시지가 역시간순으로 나오면 작업 폴더의 모든 파일이 자동으로 커밋되는 것이 아님을 확인한 것입니다.

In [3]:
# 무시 규칙과 기능 코드를 나누어 기록하고 두 커밋의 메시지를 확인합니다.
(repo / "app.py").write_text('print("주문 2건")\n', encoding="utf-8")
(repo / ".gitignore").write_text(".env\n__pycache__/\n", encoding="utf-8")
(repo / ".env").write_text("APP_MODE=demo\n", encoding="utf-8")
git("add", ".gitignore", cwd=repo)
git("commit", "-m", "chore: 생성 파일 추적 제외", cwd=repo)
git("add", "app.py", cwd=repo)
git("commit", "-m", "feat: 주문 요약 추가", cwd=repo)
print(git("status", "--short", "--ignored", cwd=repo))
print(git("log", "-2", "--pretty=%s", cwd=repo))

!! .env
feat: 주문 요약 추가
chore: 생성 파일 추적 제외


## 값 하나를 바꿔 비교

출력 숫자만 2에서 3으로 바꾸고 차이를 확인합니다. 이때 변경은 먼저 작업 폴더에만 존재합니다. `git diff -- app.py`는 작업 폴더와 색인의 차이를 보여 주므로 `-` 줄의 2와 `+` 줄의 3을 읽습니다. `git add` 뒤에는 같은 명령의 출력이 사라지고 변경이 색인으로 이동합니다. 마지막 커밋 뒤에는 작업 폴더·색인·HEAD 커밋이 다시 같은 파일 상태를 가리킵니다.

In [4]:
# 파일의 출력 숫자 하나를 바꾼 뒤 diff로 실제 변경을 읽습니다.
(repo / "app.py").write_text('print("주문 3건")\n', encoding="utf-8")
print(git("diff", "--", "app.py", cwd=repo))
git("add", "app.py", cwd=repo)
git("commit", "-m", "fix: 주문 건수 수정", cwd=repo)
print(git("log", "--oneline", "-2", cwd=repo))

diff --git a/app.py b/app.py
index 0bde456..4333761 100644
--- a/app.py
+++ b/app.py
@@ -1 +1 @@
-print("주문 2건")
+print("주문 3건")
81b0b9a fix: 주문 건수 수정
b510f0f feat: 주문 요약 추가


## 실패 재현

이미 추적 중인 파일은 나중에 무시 규칙을 추가해도 추적 상태가 유지됩니다. `*.log`에서 별표 `*`는 파일 이름의 임의 문자열과 일치하므로 `debug.log`, `server.log`처럼 `.log`로 끝나는 파일을 무시합니다.

In [5]:
# 이미 추적하는 debug.log는 무시 규칙을 추가해도 변경 대상으로 남습니다.
tracked = repo / "debug.log"
tracked.write_text("first run\n", encoding="utf-8")
git("add", "debug.log", cwd=repo)
git("commit", "-m", "chore: 디버그 로그 기록", cwd=repo)
with (repo / ".gitignore").open("a", encoding="utf-8") as file:
    file.write("*.log\n")
tracked.write_text("second run\n", encoding="utf-8")
print("추적 여부:", git("ls-files", "debug.log", cwd=repo))
print("변경 상태:", git("status", "--short", cwd=repo))

추적 여부: debug.log
변경 상태: M .gitignore
 M debug.log


## 수정

색인에서만 파일을 제거하면 작업 폴더의 파일은 남고 이후 변경은 무시됩니다.

In [6]:
# --cached는 색인에서만 제거하므로 작업 폴더의 로그 파일은 유지됩니다.
git("rm", "--cached", "debug.log", cwd=repo)
git("add", ".gitignore", cwd=repo)
git("commit", "-m", "chore: 로그 파일 추적 제외", cwd=repo)
tracked.write_text("third run\n", encoding="utf-8")
print("파일 존재:", tracked.exists())
print("추적 결과:", git("ls-files", "debug.log", cwd=repo) or "없음")
print("무시 규칙:", git("check-ignore", "-v", "debug.log", cwd=repo))
assert tracked.exists() and not git("ls-files", "debug.log", cwd=repo)

파일 존재: True
추적 결과: 없음
무시 규칙: .gitignore:3:*.log	debug.log
